# Lab 13 - Capstone: Personal Research Agent

**Section F · Capstone and Next Steps**  ·  **Estimated time:** 45–60 min  ·  **Estimated cost:** a few cents to a couple dollars (the biggest lab)

This is the **capstone**, packaged as a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

You build a Personal Research Agent: one coordinator routes the job, three specialists do the work (Researcher, Writer, Fact-Checker), two memory stores ride along (your preferences read-only, the per-topic history read-write), an outcome rubric decides when the brief is good enough, the coordinator files the cited brief to **Google Docs** via an MCP server, and the coordinator posts concise updates through the **Slack MCP**.

> **Udemy Labs note:** the connectors are real. To run the Google Docs and Slack steps you need your own Managed Agents vault IDs: `GOOGLE_DOCS_VAULT_ID` and `SLACK_VAULT_ID`. The notebook can usually read each MCP URL from the vault credential; paste `GOOGLE_DOCS_MCP_URL` or `SLACK_MCP_URL` only if a vault has multiple MCP credentials.

![Architecture](assets/architecture.svg)

## Goal
Assemble everything from the earlier labs into one end-to-end agentic system. You will wire a multi-agent roster behind a single coordinator, attach two memory stores with different access modes, declare Google Docs and Slack MCP servers on the coordinator through vaults, drive the run with an outcome rubric until the brief is genuinely good enough, file the result to Google Docs, post Slack updates, and pull the finished brief back locally.

## Prerequisites
- A working setup from the earlier labs: multi-agent rosters, memory stores, outcome rubrics, and MCP via a vault should all feel familiar.
- An Anthropic API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- A Google Docs Managed Agents vault id: paste `GOOGLE_DOCS_VAULT_ID` in the setup cell. The notebook can usually read `GOOGLE_DOCS_MCP_URL` from the vault credential; paste the URL only if the vault has multiple MCP credentials.
- A Slack Managed Agents vault id: paste `SLACK_VAULT_ID` in the setup cell. The notebook can usually read `SLACK_MCP_URL` from the vault credential; paste the URL only if the vault has multiple MCP credentials.
- A target Slack channel such as `#research`. This is a routing setting, not a secret.


In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

if not os.environ.get("GOOGLE_DOCS_VAULT_ID"):
    os.environ["GOOGLE_DOCS_VAULT_ID"] = getpass.getpass(
        "Enter your Google Docs Managed Agents vault ID (vlt_...): "
    ).strip()

if not os.environ.get("SLACK_VAULT_ID"):
    os.environ["SLACK_VAULT_ID"] = getpass.getpass(
        "Enter your Slack Managed Agents vault ID (vlt_...): "
    ).strip()

if not os.environ.get("SLACK_CHANNEL"):
    os.environ["SLACK_CHANNEL"] = input("Slack channel for agent updates [#research]: ").strip() or "#research"

# Optional advanced overrides only. In the normal course path, leave these unset;
# the next cell reads the MCP URLs from the vault credentials.
# os.environ["GOOGLE_DOCS_MCP_URL"] = "https://your-google-docs-mcp.example.com/mcp"
# os.environ["SLACK_MCP_URL"] = "https://your-slack-mcp.example.com/mcp"

print("ANTHROPIC_API_KEY configured for this notebook session.")
print("GOOGLE_DOCS_VAULT_ID configured for this notebook session.")
print("SLACK_VAULT_ID configured for this notebook session.")
print("SLACK_CHANNEL =", os.environ["SLACK_CHANNEL"])


In [ ]:
import os, re
from pathlib import Path
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

# The managed-agents beta header is required across the beta.* calls below
# (including files.list / files.download at the end).
BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, coordinator model:", MODEL)

### Pick a topic
The whole pipeline runs against one topic. Change this and re-run from here to research something else (the per-topic memory store keys off the slug).

In [ ]:
def slugify(text):
    """Turn a topic into a filesystem- and store-name-safe slug."""
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:50] or "topic"

TOPIC = "small modular reactors"
SLUG = slugify(TOPIC)
print("topic =", TOPIC, "| slug =", SLUG)

### Step 1 - Build the environment and resolve the vaults
Create a cloud environment with **limited** networking that explicitly allows the Google Docs endpoints and MCP servers, then resolve the Google Docs and Slack connections from their vaults. The vaults let the agent file the brief *as you* and post to Slack *as you*: tokens never reach the sandbox where Claude's code runs.


In [ ]:
from urllib.parse import urlparse


def validate_vault_id(vault_id, env_var="GOOGLE_DOCS_VAULT_ID", provider="Google Docs"):
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError(f"Set {env_var} to the Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            f"{env_var} currently contains an Anthropic API key. Paste the "
            f"{provider} vault id from Claude Console instead; it should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"{env_var} should start with 'vlt_'. Got: {vault_id!r}")


def validate_mcp_url(url, url_env="GOOGLE_DOCS_MCP_URL", vault_env="GOOGLE_DOCS_VAULT_ID"):
    """Catch missing or placeholder URLs before agent creation."""
    parsed = urlparse(url)
    if not url or "REPLACE-ME" in url or parsed.scheme != "https" or not parsed.netloc:
        raise RuntimeError(
            f"Set {url_env} to a valid https MCP endpoint, or use a "
            f"{vault_env} whose credential contains an MCP server URL."
        )


def mcp_url_from_vault(vault_id, provider):
    """Read an MCP URL from a provider credential in a Managed Agents vault."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(getattr(credential, "auth", None), "type", None) == "mcp_oauth"
    ]
    provider_credentials = [
        credential for credential in mcp_credentials
        if provider.lower() in (
            f"{getattr(credential, 'display_name', '') or ''} "
            f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '')}"
        ).lower()
    ]

    if len(provider_credentials) == 1:
        return provider_credentials[0].auth.mcp_server_url
    if len(mcp_credentials) == 1:
        return mcp_credentials[0].auth.mcp_server_url

    names = [
        f"{getattr(credential, 'display_name', '') or credential.id}: "
        f"{getattr(getattr(credential, 'auth', None), 'mcp_server_url', '<no mcp url>')}"
        for credential in mcp_credentials
    ]
    raise RuntimeError(
        f"Could not uniquely identify the {provider} MCP credential in "
        f"vault {vault_id}. Set the matching MCP URL env var explicitly. "
        f"Found MCP credentials: {names or 'none'}"
    )


def resolve_required_mcp_connection(provider, vault_env, url_env):
    """Return (vault_id, mcp_url) for a required vault-backed MCP connection."""
    vault_id = os.environ.get(vault_env, "").strip()
    mcp_url = os.environ.get(url_env, "").strip()
    validate_vault_id(vault_id, vault_env, provider.title())
    mcp_url = mcp_url or mcp_url_from_vault(vault_id, provider)
    validate_mcp_url(mcp_url, url_env, vault_env)
    os.environ[url_env] = mcp_url
    print(f"{provider} vault =", vault_id, "(existing vault)")
    print(f"{provider} MCP URL =", mcp_url)
    return vault_id, mcp_url


env = client.beta.environments.create(
    name="capstone-env",
    config={
        "type": "cloud",
        "packages": {"pip": ["beautifulsoup4", "markdownify"]},
        "networking": {
            "type": "limited",
            "allowed_hosts": ["docs.googleapis.com", "www.googleapis.com"],
            "allow_mcp_servers": True,
            "allow_package_managers": True,
        },
    },
    betas=BETAS,
)
print("environment =", env.id)

google_docs_vault_id, GOOGLE_DOCS_MCP_URL = resolve_required_mcp_connection(
    provider="google",
    vault_env="GOOGLE_DOCS_VAULT_ID",
    url_env="GOOGLE_DOCS_MCP_URL",
)
slack_vault_id, SLACK_MCP_URL = resolve_required_mcp_connection(
    provider="slack",
    vault_env="SLACK_VAULT_ID",
    url_env="SLACK_MCP_URL",
)
SLACK_CHANNEL = os.environ.get("SLACK_CHANNEL", "#research").strip() or "#research"


### Step 2 - Configure the `multiagent.agents` roster
Create the three specialists, then create the coordinator with the roster listing them. All agents use `claude-haiku-4-5-20251001`; role differences come from prompts, tool scope, and the coordinator topology.

The system prompts hand off artifacts as files under `/workspace`, read user preferences from the read-only memory store, and append to the read-write topic store.

In [ ]:
RESEARCHER = """You research a topic. Use web_search for breadth and
web_fetch on the most promising results. Prefer the sources listed in
/mnt/memory/user-prefs/trusted_sources.md. Write a JSON array of citations
to /workspace/citations.json: each item {url, title, snippet, why_relevant}.
Aim for at least 5 distinct, high-quality sources."""

WRITER = """You draft a research brief from the researcher's citations.
Read /workspace/citations.json for sources and
/mnt/memory/user-prefs/style.md for tone and length. Produce
/workspace/draft.md: 500-600 words, every non-trivial claim carrying an
inline citation link. Do not invent claims or sources."""

FACT_CHECKER = """For every claim in /workspace/draft.md, verify it against
the linked source with web_fetch. Write /workspace/check.md, marking each
claim:
  [verified]   - quote the supporting line from the source
  [partial]    - quote the source and explain the gap
  [unverified] - explain why it could not be confirmed
If any claim is [partial] or [unverified], the writer must revise."""

COORDINATOR = """You coordinate a Personal Research Agent team. At the start
of every session, read /mnt/memory/user-prefs/style.md and
/mnt/memory/user-prefs/trusted_sources.md so the brief matches the user's
preferences.

Given a topic:
1) Delegate to the Researcher (return when /workspace/citations.json exists).
2) Delegate to the Writer (return when /workspace/draft.md exists).
3) Delegate to the Fact-Checker (return when /workspace/check.md exists).
4) If check.md contains any [partial] or [unverified] claim, loop steps 2-3
   with the Writer fixing those claims, until the draft is clean.
5) Save the final brief to /mnt/session/outputs/brief.md.
6) Post at most two concise updates to the configured Slack channel through
   the slack MCP server: one after research starts and one final completion
   message after the Google Doc is filed. Do not post drafts or long content.
7) File the brief as a new Google Doc under a "Research" folder using the
   google_docs MCP server. Keep the document title equal to the topic.
8) Append a one-line summary of this brief to
   /mnt/memory/topic-context/<topic-slug>/log.md so future runs recall it."""


In [ ]:
researcher = client.beta.agents.create(
    name="Capstone Researcher",
    model="claude-haiku-4-5-20251001",
    system=RESEARCHER,
    tools=[{
        "type": "agent_toolset_20260401",
        "default_config": {"enabled": False},
        "configs": [
            {"name": "web_search", "enabled": True},
            {"name": "web_fetch", "enabled": True},
            {"name": "read", "enabled": True},
            {"name": "write", "enabled": True},
        ],
    }],
    betas=BETAS,
)

writer = client.beta.agents.create(
    name="Capstone Writer",
    model="claude-haiku-4-5-20251001",
    system=WRITER,
    # Drafting is low risk, so the full built-in toolset is fine here.
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)

fact_checker = client.beta.agents.create(
    name="Capstone Fact-Checker",
    model="claude-haiku-4-5-20251001",  # verification is where errors are most expensive
    system=FACT_CHECKER,
    tools=[{
        "type": "agent_toolset_20260401",
        "default_config": {"enabled": False},
        "configs": [
            {"name": "web_fetch", "enabled": True},
            {"name": "read", "enabled": True},
            {"name": "write", "enabled": True},
        ],
    }],
    betas=BETAS,
)

print("specialists:", researcher.id, writer.id, fact_checker.id)

### Step 2 (cont.) + Step 4 - Create the coordinator and declare the MCP servers
The **Google Docs and Slack MCP servers live on the coordinator only**; the specialists never touch them. The coordinator authenticates to both through vault credentials at session time. `multiagent.agents` lists the roster it can delegate to.


In [ ]:
coordinator_system = (
    COORDINATOR
    + f"\n\nSlack target channel: {SLACK_CHANNEL}. "
    "Use the slack MCP server for the two short status updates only."
)

coordinator = client.beta.agents.create(
    name="Capstone Research Lead",
    model=MODEL,  # claude-haiku-4-5-20251001
    system=coordinator_system,
    mcp_servers=[
        {"type": "url", "name": "google_docs", "url": GOOGLE_DOCS_MCP_URL},
        {"type": "url", "name": "slack", "url": SLACK_MCP_URL},
    ],
    tools=[
        {"type": "agent_toolset_20260401"},
        {"type": "mcp_toolset", "mcp_server_name": "google_docs"},
        {
            "type": "mcp_toolset",
            "mcp_server_name": "slack",
            "default_config": {"permission_policy": {"type": "always_allow"}},
        },
    ],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "agent", "id": researcher.id},
            {"type": "agent", "id": writer.id},
            {"type": "agent", "id": fact_checker.id},
        ],
    },
    betas=BETAS,
)
print("coordinator =", coordinator.id)


### Step 3 - Attach the two memory stores
Create a `user-prefs` store and seed it once with tone, length, and trusted sources. Create (or reuse) a per-topic `topic-context` store. You attach both to the session: `user-prefs` as `read_only` so preferences can never be overwritten, and `topic-context` as `read_write` so the coordinator can append a running log that accumulates across runs.

In [ ]:
prefs = client.beta.memory_stores.create(
    name="capstone-user-prefs",
    description="User preferences for research briefs (tone, length, sources).",
    betas=BETAS,
)
client.beta.memory_stores.memories.create(
    prefs.id,
    path="/style.md",
    content=(
        "Tone: concise, analytical, no fluff.\n"
        "Length: 500-600 words.\n"
        "Avoid bullet lists in the body; use them only for the sources list."
    ),
    betas=BETAS,
)
client.beta.memory_stores.memories.create(
    prefs.id,
    path="/trusted_sources.md",
    content=(
        "Prefer: anthropic.com, arxiv.org, stanford.edu, mit.edu, "
        "nature.com, the IEA, and official government sources.\n"
        "Avoid: clickbait, press-release rewrites, anonymous blogs."
    ),
    betas=BETAS,
)
print("user-prefs store =", prefs.id)

# Reuse the per-topic store across runs so it accumulates context.
topic_name = f"capstone-topic-{SLUG}"
existing = [s for s in client.beta.memory_stores.list(betas=BETAS).data if s.name == topic_name]
if existing:
    topic_store = existing[0]
    print("reusing topic store =", topic_store.id)
else:
    topic_store = client.beta.memory_stores.create(
        name=topic_name, description=f"Per-topic research log for: {TOPIC}", betas=BETAS,
    )
    print("created topic store =", topic_store.id)

### The outcome rubric
The rubric is the loop's exit condition. It encodes both goals at once: the brief must be well-cited and within the length budget. It also points the grader at the trusted-sources memory file and the Google Docs filing.

In [ ]:
RUBRIC = """# Research Brief Rubric

## Content
- Topic clearly defined in the opening paragraph.
- 4-6 substantive claims, each supported by an inline citation.
- No claim without a source. 500-600 words total.

## Sources
- At least 5 distinct sources.
- All from /mnt/memory/user-prefs/trusted_sources.md or comparable quality
  (peer-reviewed, official, or a well-known publication).
- Each source linked by URL in an inline citation.

## Output
- Saved as /mnt/session/outputs/brief.md.
- Filed to a new Google Doc under "Research" via the google_docs MCP server.
"""
print(RUBRIC)

### Step 5 - Run an outcome-driven session with the rubric
Create the session with both vaults and both memory stores attached, then send `user.define_outcome` with the rubric and a `max_iterations` safety rail. One call kicks off the whole pipeline: research, draft, fact-check, grade, Slack updates, Google Docs filing, and loop on a failed grade.


In [ ]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": coordinator.id},
    environment_id=env.id,
    vault_ids=[google_docs_vault_id, slack_vault_id],
    resources=[
        {"type": "memory_store", "memory_store_id": prefs.id, "access": "read_only"},
        {"type": "memory_store", "memory_store_id": topic_store.id, "access": "read_write"},
    ],
    title=TOPIC,
    betas=BETAS,
)
print("session.id =", session.id, "\n")

satisfied = False
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{
            "type": "user.define_outcome",
            "description": (
                f"Produce a research brief on: {TOPIC}. "
                f"Run it through the Personal Research Agent pipeline, "
                f"file the result to Google Docs, and post concise progress "
                f"and completion updates to Slack channel {SLACK_CHANNEL}."
            ),
            "rubric": {"type": "text", "content": RUBRIC},
            "max_iterations": 8,
        }],
        betas=BETAS,
    )
    for event in stream:
        if event.type == "session.thread_created":
            print(f"+ thread {event.agent_name}")
        elif event.type == "agent.thread_message_received":
            print(f"  <- {event.from_agent_name} returned")
        elif event.type == "agent.mcp_tool_use":
            print(f"  [mcp: {event.name}]")
        elif event.type == "span.outcome_evaluation_end":
            print(f"  iter {event.iteration}: {event.result}")
            satisfied = event.result == "satisfied"
        elif event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

print("\nsatisfied =", satisfied)


### Step 6 - Retrieve the brief
List the session's files, find `brief.md`, and download it locally so you have the cited brief next to the Google Doc.


In [ ]:
out_dir = Path("./outputs")
out_dir.mkdir(exist_ok=True)
saved = False
for f in client.beta.files.list(scope_id=session.id, betas=BETAS):
    if f.filename == "brief.md":
        target = out_dir / f"{SLUG}.md"
        client.beta.files.download(f.id, betas=BETAS).write_to_file(str(target))
        print("saved:", target)
        saved = True
if not saved:
    print("No brief.md found in the session files (did the run reach a satisfied outcome?).")

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
The stream shows the specialists spawn and return, the grader's iteration count climb toward `satisfied`, and the coordinator's MCP calls into Google Docs and Slack. The run ends with a cited brief filed to a new Google Doc and short Slack updates posted by the agent.

```
session.id = sesn_01...

+ thread Capstone Researcher
  <- Capstone Researcher returned
+ thread Capstone Writer
  <- Capstone Writer returned
+ thread Capstone Fact-Checker
  <- Capstone Fact-Checker returned
  [mcp: post_message]
  iter 1: not_satisfied
+ thread Capstone Writer
  <- Capstone Writer returned
  iter 2: satisfied
  [mcp: create_document]
  [mcp: post_message]
--- session idle ---

satisfied = True
saved: outputs/small-modular-reactors.md
```

- A new Google Doc under "Research" with the cited brief as its content.
- A short Slack progress/final update in your configured channel.
- `outputs/<topic-slug>.md` saved locally.


## Troubleshooting
- **No Google Doc filed.** Confirm the vault has a Google Docs MCP OAuth credential and that the resolved MCP URL is the one declared on the coordinator. Also confirm `docs.googleapis.com` is in the environment's `allowed_hosts`.
- **No Slack update.** Confirm the vault has a Slack MCP credential, `SLACK_CHANNEL` points to a channel the connected Slack account can post to, and the run reached the Slack step. If the channel is private, invite the connected Slack app/user.
- **`failed` or stuck outcome.** Re-read the rubric against the topic description; if they contradict, the grader can never pass. The Fact-Checker may also be failing on weak sources, so the loop never goes clean. Soften the rubric or strengthen the Researcher's trusted-sources preferences.
- **MCP auth errors.** If you see `mcp_auth_failed`, the Google Docs or Slack credential in the vault is missing, expired, scoped without the required permission, or registered for a different MCP URL. Reconnect the credential in the vault. Refresh fields are optional hardening for longer-lived Google runs.
- **`usage` / cost questions.** This is the biggest lab; the multi-agent roster and multi-iteration loop are the main cost drivers. Lowering `max_iterations` is the main cost dial.


## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste these prompts in order. Claude Code can author the script and then run it against your connectors.

1. Scaffold the agents and infrastructure:

> Write a Python script `lab13.py` using the Anthropic SDK `client.beta.*` Agents APIs (`betas=['managed-agents-2026-04-01']`). Create a cloud environment with limited networking that allows `docs.googleapis.com` and `www.googleapis.com` plus MCP servers and package managers. Use existing Managed Agents vaults from `GOOGLE_DOCS_VAULT_ID` and `SLACK_VAULT_ID`; read each MCP URL from the vault when possible, with `GOOGLE_DOCS_MCP_URL` and `SLACK_MCP_URL` as optional overrides. Create three specialist agents (a Researcher and a Writer on `claude-haiku-4-5-20251001`, a Fact-Checker on `claude-haiku-4-5-20251001`) each with the `agent_toolset_20260401` toolset scoped to the tools that role needs.

2. Wire the coordinator, memory, and outcome loop:

> In the same script, create a coordinator agent named "Capstone Research Lead" on `claude-haiku-4-5-20251001` with a `multiagent.agents` roster of the three specialists, `mcp_toolset` entries for `google_docs` and `slack`, and a system prompt that delegates research, drafting, and fact-checking, loops on fact-check failures, files the final brief to a new Google Doc under "Research", posts at most two concise Slack updates to `SLACK_CHANNEL`, and logs a one-line summary to the topic memory store. Create a `user-prefs` memory store (seed it with tone, length, and trusted sources) and a per-topic `topic-context` store. Create a session attaching both vaults, `user-prefs` as read_only, and `topic-context` as read_write. Send `user.define_outcome` with a rubric requiring a well-cited 500-600 word brief filed to Google Docs, with `max_iterations=8`. Stream the run and print thread, MCP, and outcome-evaluation events.

3. Retrieve and run:

> After the run, download the session's `brief.md` to `./outputs/<topic-slug>.md`. Run `python lab13.py "small modular reactors"` with my env vars set and show me the streamed events, the Google Doc that was filed, and the Slack MCP update. If either external action did not appear, read the error and tell me whether the vault credential or the MCP URL is the problem.

Compare what Claude Code writes to the cells above. The committed `lab13.py` in this folder is one reference answer.


## Stretch
- **Run a second topic.** Change `TOPIC` to something new and run, then switch back to the first topic and run again. Confirm the `topic-context` store recalls the earlier run while `user-prefs` is reused unchanged.
- **Add another MCP target.** Attach a third vault-backed MCP server, such as Linear or GitHub, and ask the coordinator to file follow-up work after the brief is complete.
- **Cost-optimize.** Lower `max_iterations`, tighten the rubric, or scope specialist tools more narrowly and measure quality on a few briefs.
- **Consolidate memory.** After several briefs, run a consolidation pass over the `topic-context` stores so retrieval stays cheap, then point the next run at the consolidated store.

## What you've shipped
A real, end-to-end agentic system that coordinates multiple Claude specialists, remembers and improves across sessions, reaches into real third-party services securely through vaults, drives itself to a quality bar with an outcome rubric, and announces its own results. This is the portfolio piece.
